In [11]:
import uproot
import dask_awkward as dak
import awkward as ak
import numpy as np
import pandas as pd
import vector
from dask.diagnostics import ProgressBar
import os

In [2]:
vector.register_awkward()

In [3]:
# ==============================================================================
# 1. TRUTH MATCHING (Filtro de Eventos e Identificação de Partículas)
# ==============================================================================
def apply_truth_match(df):
    """
    Recebe o 'dataframe' Awkward, identifica os eventos de sinal,
    e FILTRA as partículas de cada evento mantendo APENAS as que pertencem ao sinal.
    """
    pdg = df["BPHGenPart_pdgId"]
    mother = df["BPHGenPart_genPartIdxMother"]
    local_idx = ak.local_index(pdg)

    safe_mom = ak.where(mother >= 0, mother, 0)
    mom_pdg  = pdg[safe_mom]
    mom_pdg  = ak.where(mother >= 0, mom_pdg, -999)

    safe_grandmom = safe_mom[safe_mom]
    grandmom_pdg  = pdg[safe_grandmom]
    grandmom_pdg  = ak.where(safe_mom >= 0, grandmom_pdg, -999)

    B_ancestor_idx = ak.full_like(pdg, -1)
    B_ancestor_idx = ak.where((pdg == 511), local_idx, B_ancestor_idx)
    B_ancestor_idx = ak.where((pdg == 313) & (mom_pdg == 511), safe_mom, B_ancestor_idx)
    B_ancestor_idx = ak.where((pdg == 22)  & (mom_pdg == 511), safe_mom, B_ancestor_idx)

    is_K    = (pdg == 321)  & (mom_pdg == 313) & (grandmom_pdg == 511)
    is_pi   = (pdg == -211) & (mom_pdg == 313) & (grandmom_pdg == 511)
    is_mu_m = (pdg == 13)   & (mom_pdg == 22)  & (grandmom_pdg == 511)
    is_mu_p = (pdg == -13)  & (mom_pdg == 22)  & (grandmom_pdg == 511)

    B_ancestor_idx = ak.where(is_K,               safe_grandmom, B_ancestor_idx)
    B_ancestor_idx = ak.where(is_pi,              safe_grandmom, B_ancestor_idx)
    B_ancestor_idx = ak.where(is_mu_m | is_mu_p,  safe_grandmom, B_ancestor_idx)

    anc_kstar = B_ancestor_idx[(pdg == 313) & (mom_pdg == 511)]
    anc_K     = B_ancestor_idx[is_K]
    anc_pi    = B_ancestor_idx[is_pi]
    anc_mu_m  = B_ancestor_idx[is_mu_m]
    anc_mu_p  = B_ancestor_idx[is_mu_p]

    p_anc = B_ancestor_idx[:, :, np.newaxis]

    has_1_kstar = ak.sum(p_anc == anc_kstar[:, np.newaxis, :], axis=-1) == 1
    has_1_K     = ak.sum(p_anc == anc_K[:, np.newaxis, :],     axis=-1) == 1
    has_1_pi    = ak.sum(p_anc == anc_pi[:, np.newaxis, :],    axis=-1) == 1
    has_1_mu_m  = ak.sum(p_anc == anc_mu_m[:, np.newaxis, :],  axis=-1) == 1
    has_1_mu_p  = ak.sum(p_anc == anc_mu_p[:, np.newaxis, :],  axis=-1) == 1

    is_valid_chain  = has_1_kstar & has_1_K & has_1_pi & has_1_mu_m & has_1_mu_p
    particle_mask   = is_valid_chain & (B_ancestor_idx >= 0)
    event_has_signal = ak.sum(particle_mask, axis=1) > 0

    # Sobrescreve as colunas BPHGenPart_ mantendo APENAS as partículas do sinal
    for col in df.fields:
        if col.startswith("BPHGenPart_"):
            df[col] = df[col][particle_mask]

    # Retorna apenas os eventos que possuem o sinal
    return df[event_has_signal]

In [4]:
# ==============================================================================
# 2. SELEÇÃO E ACEITAÇÃO
# ==============================================================================
def apply_selection(df):
    """
    Aplica cortes cinemáticos. Como aplicamos a máscara no passo anterior, 
    todos os pdgs, pts e etas já são exclusivamente do decaimento de sinal.
    """
    pdg = df["BPHGenPart_pdgId"]
    pt  = df["BPHGenPart_pt"]
    eta = df["BPHGenPart_eta"]

    # Identifica múons e hádrons
    is_mu = (abs(pdg) == 13)
    is_h  = ((abs(pdg) == 321) | (abs(pdg) == 211))

    # Aplica os cortes 
    mu_pass = is_mu & (pt > 2.0) & (abs(eta) < 2.4)
    h_pass  = is_h  & (pt > 1.5) & (abs(eta) < 2.4)

    # O evento só sobrevive se os 2 múons e os 2 hádrons passaram no corte
    event_pass = (ak.sum(mu_pass, axis=1) == 2) & (ak.sum(h_pass, axis=1) == 2)

    return df[event_pass]

In [5]:
# ==============================================================================
# 3. CÁLCULO DE MASSA INVARIANTE (q^2)
# ==============================================================================
def calculate_q2(df):
    """
    Constrói os quadrivetores dos múons e calcula a massa invariante q^2.
    """
    # Identifica os múons (que agora sabemos que são do sinal)
    is_mu = (abs(df["BPHGenPart_pdgId"]) == 13)

    # Empacota no formato que a biblioteca vector reconhece
    muons_4v = ak.zip(
        {
            "pt":   df["BPHGenPart_pt"][is_mu],
            "eta":  df["BPHGenPart_eta"][is_mu],
            "phi":  df["BPHGenPart_phi"][is_mu],
            "mass": df["BPHGenPart_mass"][is_mu]
        },
        with_name="Momentum4D"
    )

    # Soma os dois múons (índice 0 e índice 1 de cada evento)
    dimuon = muons_4v[:, 0] + muons_4v[:, 1]

    # Cria a coluna q2 no dataframe
    df["q"] = dimuon.mass
    return df

In [ ]:
# ==============================================================================
# 4. EXECUÇÃO DO PIPELINE (O "Main")
# ==============================================================================
file_path = "/cms/store/user/tdeandra/BPH_NanoAOD_MC/BdtoKstar2Mu_KstartoKPi_TuneCP5_13p6TeV_pythia8-evtgen/BPH_Nano_MC_BdtoKstarMuMu_NoFilter_2022/260427_132612/0000/*.root"

print("Lendo arquivos e montando o grafo Lazy...")
df = uproot.dask(
    {file_path: "Events"},
    library="ak"
)

In [ ]:
df = apply_truth_match(df)
df = apply_selection(df)
df = calculate_q2(df)

In [13]:
features = [
    "BPHGenPart_pt",
    "BPHGenPart_eta",
    "BPHGenPart_phi",
    "BPHGenPart_mass",
    "q",
    "BPHGenPart_pdgId"
]

In [14]:
with ProgressBar():
    B = df[features].compute()    

NameError: name 'df' is not defined

In [15]:
df_pandas = ak.to_dataframe(B)
df_pandas = df_pandas.reset_index()

NameError: name 'B' is not defined

In [16]:
df_pandas['q2'] = df_pandas['q']**2

bins = [1.1, 2, 4.3, 6, 8.68, 10.09, 12.86, 14.18, 16, 19]
labels = [0, 1, 2, 3, 4, 5, 6, 7, 8]

df_pandas['bin_index'] = pd.cut(df_pandas['q2'], bins=bins, labels=labels, right=False)
df_pandas['bin_index'] = pd.cut(df_pandas['q2'], bins=bins, labels=labels, right=False).astype(float)

NameError: name 'df_pandas' is not defined

In [17]:
is_B0 = df_pandas["BPHGenPart_pdgId"] == 511

print(f"Total de partículas no DF Gen: {len(df_pandas)}")

for i in range(9):  
    is_bin = df_pandas["bin_index"] == i    
    df_filtrado = df_pandas[is_B0 & is_bin]    
    print(f"Bin {i}: {len(df_filtrado)} partículas B0 encontradas")

NameError: name 'df_pandas' is not defined